In [2]:
import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader , TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [15]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to C:\Users\sonu
[nltk_data]     singh\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to C:\Users\sonu
[nltk_data]     singh\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to C:\Users\sonu
[nltk_data]     singh\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [3]:
df = pd.read_csv("IMDB Dataset.csv")

In [4]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [7]:
df.drop_duplicates(inplace=True)

In [8]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


# preprocessing and cleaning the data

## cleaning data

In [12]:
def cleaning_data(text):
    # lower casing 
    text = text.lower()
    # removing html tags
    text = re.sub(r"<.*?>","",text)
    # removing urls https+
    text = re.sub(r"http/S+","",text)
    # removing puncutations
    text = re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
df['review']=df['review'].apply(cleaning_data)

## removing stopwords 

In [16]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")
    return text
df['review']=df['review'].apply(remove_stopwords)

In [17]:
from nltk.stem import PorterStemmer 

## Stemming 

In [19]:
def stemming(text):
    tokens = word_tokenize(text)
    new_tokens = []
    pe = PorterStemmer()
    for word in tokens:
        new_tokens.append(pe.stem(word))
    return " ".join(new_tokens)
df['review']=df['review'].apply(stemming)

## Vectorization - Embeddings

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [22]:
tf = TfidfVectorizer(max_features = 5000)
X = tf.fit_transform(df['review'])

In [24]:
y = df['sentiment']
le = LabelEncoder()
y=le.fit_transform(y)

## train_test_spliting 

In [26]:
X_train,X_test,Y_train,Y_test = train_test_split(
    X,y,random_state=42,test_size=0.2
)

### compressed matrix --> numpy array --> tensor dataset --> dataloader

In [29]:
# compressed matrix --> numpy array 
X_train = X_train.toarray()
X_test = X_test.toarray()

# numpy array --> tensor dataset 
train_dataset = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(Y_train).float()
)
test_dataset = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(Y_test).float()
)

In [32]:
trainloader = DataLoader(train_dataset,batch_size = 64,shuffle = True)
testloader = DataLoader(test_dataset,batch_size = 64,shuffle = True)

## LSTM Architecture

In [33]:
class Lstm(nn.Module):
    def __init__(self,input_size,hidden_size = 128 , num_layer = 1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layer = num_layer
        
        self.lstm = nn.LSTM(input_size,hidden_size,num_layer,batch_first = True)

        self.fc = nn.Linear(in_features = hidden_size,out_features=1)

    def forward(self,x):
        
        # hidden state  -> (num_layer , batch_size , hidden_size)
        h0 = torch.zeros(self.num_layer,x.size(0),self.hidden_size)

        # cell state -> (num_layer , batch_size , hidden_size)
        c0 = torch.zeros(self.num_layer,x.size(0),self.hidden_size)

        out , _ = self.lstm(x,(h0,c0))
        # out -> hidden state of all feature row --> (batch_size , seq_layer , hidden_size)
        # _ -> final hidden state of all dataset 

        out = self.fc(out[:,-1,:])
        return out

In [37]:
input_size = X_train.shape[1]  # 50000  --> 128 --> 1 
model = Lstm(input_size)
model.to(device)
optimizer = optim.Adam(model.parameters())
criterion = nn.BCELoss()

## training the model 

In [39]:
epochs = 10
for epoch in range(epochs):
    model.train()
    for xb , yb in trainloader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        # NOTE -> xb (batch_size , 5000)     but lstm model need (batch_size , seq_layer , 5000)
        xb = xb.unsqueeze(1)
        output = model(xb) # (batch_size , 1)
        output = torch.sigmoid(output.squeeze()) # (batch_size,)
        loss = criterion(output,yb)
        loss.backward()
        optimizer.step()
    print(f'for {epoch}/{epochs} loss = {loss}')

for 0/10 loss = 0.24071751534938812
for 1/10 loss = 0.23682938516139984
for 2/10 loss = 0.4820431172847748
for 3/10 loss = 0.26364338397979736
for 4/10 loss = 0.2761516571044922
for 5/10 loss = 0.20283353328704834
for 6/10 loss = 0.33079248666763306
for 7/10 loss = 0.18232421576976776
for 8/10 loss = 0.17381197214126587
for 9/10 loss = 0.22990232706069946


In [40]:
torch.save(model.state_dict(),"sentimental_analysis.pth")

##  evalution 

In [43]:
model.load_state_dict(torch.load("sentimental_analysis.pth"))
model.eval()
with torch.no_grad():
    correct_vals = 0.0
    total_vals = 0.0
    for xb,yb in testloader:
        xb = xb.unsqueeze(1)
        output = model(xb)
        output = (torch.sigmoid(output.squeeze())>0.5).int()  # if torch.sigmoid(output) > 0.5  = true -> 1    , false --> 0
        correct_vals += (output == yb).sum().item()
        total_vals += yb.size(0)  # 64 + 64 ...... len(testloader) = 64*155 
    print(f'accuracy = {(correct_vals/total_vals)*100}')

accuracy = 85.74165574266411


## tesing for new review

In [77]:
review = "While visually immaculate, Oppenheimer ultimately suffocates under the weight of its own self-importance. Clocking in at three hours, the film feels like a relentlessly loud, over-edited montage of men in rooms talking rapidly about physics and politics. Because Nolan favors historical grandiosity over human connection, the characters feel like flat historical chess pieces rather than real people, making it incredibly difficult to emotionally connect with anyone on screen. It is an impressive exercise in cinematic scale, but it is an emotionally hollow and exhausting watch."
review = cleaning_data(review)
review = remove_stopwords(review)
review = stemming(review)
print(review)

vulli immcul oppenheim ultim suffoc weight selfimportnc clockg three hour film feel like relentlessli loud ede mtge men room tlkg rpidli bout physic nd polic becus noln fvor hricl grndiosi humn cnecti chrcter feel like fl hricl chess piec rr thn rel peopl mkg credibl difficult emotilli cnect wh nye screen n impress exerc cemic scle n emotilli hollow nd exhustg wch


In [78]:
X = tf.transform([review])

In [79]:
x=torch.tensor(
    X.toarray(),
    dtype = torch.float32
)

In [80]:
with torch.no_grad():
    x = x.unsqueeze(1)
    output = model(x)
    output = (torch.sigmoid(output.squeeze())>0.5).float()

    if output == 0:
        print("sentiment is negative")
    else:
        print("sentiment is positive")

sentiment is negative
